In [2]:
# Load libraries needed to recompute and validate the published metrics.
from pathlib import Path
import pandas as pd
import numpy as np

In [ ]:
# Load cleaned tables and prepare product prices and costs for validation.
DATA_DIR = Path("../data/cleaned")
REPORT_DIR = Path("../reports")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# loading all the cleaned data
sales = pd.read_csv(DATA_DIR / "Sales.csv")
products= pd.read_csv(DATA_DIR / "Products.csv")
stores= pd.read_csv(DATA_DIR / "Stores.csv")
customers= pd.read_csv(DATA_DIR / "Customers.csv")

sales["Order Date"] = pd.to_datetime(
    sales["Order Date"], errors="coerce"
)

for column in ["Unit Price USD", "Unit Cost USD"]:
    products[column] = pd.to_numeric(
        products[column], errors="coerce"
    )

# Validate dimension key uniqueness
assert products["ProductKey"].is_unique

# Merge product information without multiplying sales lines
df = sales.merge(
    products[
        ["ProductKey", "Unit Price USD", "Unit Cost USD"]
    ],
    on="ProductKey",
    how="left",
    validate="many_to_one",
    indicator=True
)

assert len(df) == len(sales)

unmatched_products = int((df["_merge"] != "both").sum())

df["Revenue USD"] = df["Quantity"] * df["Unit Price USD"]
df["COGS USD"] = df["Quantity"] * df["Unit Cost USD"]
df["Gross Profit USD"] = df["Revenue USD"] - df["COGS USD"]

revenue = df["Revenue USD"].sum()
cogs = df["COGS USD"].sum()
profit = df["Gross Profit USD"].sum()
orders = df["Order Number"].nunique()
units = df["Quantity"].sum()

results = pd.DataFrame({
    "Metric": [
        "Revenue USD",
        "COGS USD",
        "Gross Profit USD",
        "Gross Margin %",
        "Distinct Orders",
        "Units Sold",
        "Average Order Value USD",
        "Unmatched Product Records"
    ],
    "Value": [
        revenue,
        cogs,
        profit,
        profit / revenue if revenue else np.nan,
        orders,
        units,
        revenue / orders if orders else np.nan,
        unmatched_products
    ]
})

print(results.to_string(index=False))

results.to_csv(
    REPORT_DIR / "validation_results.csv",
    index=False
)

                   Metric        Value
              Revenue USD 5.575548e+07
                 COGS USD 2.309279e+07
         Gross Profit USD 3.266269e+07
           Gross Margin % 5.858202e-01
          Distinct Orders 2.632600e+04
               Units Sold 1.977570e+05
  Average Order Value USD 2.117886e+03
Unmatched Product Records 0.000000e+00


In [8]:
# Recalculate category results and compare them with the analysis outputs.
category_check = (
    df.merge(
        pd.read_csv(DATA_DIR / "Products.csv")[
            ["ProductKey", "Category"]
        ],
        on="ProductKey",
        how="left",
        validate="many_to_one"
    )
    .groupby("Category", as_index=False)
    .agg(
        Revenue=("Revenue USD", "sum"),
        Gross_Profit=("Gross Profit USD", "sum")
    )
    .sort_values("Revenue", ascending=False)
)

display(category_check)

,Category,Revenue,Gross_Profit
3,Computers,19301595.46,11277447.90
5,Home Appliances,10795478.59,6296338.85
1,Cameras and camcorders,6520168.02,3919800.99
2,Cell phones,6183791.22,3498626.54
7,TV and Video,5928982.69,3536694.39
0,Audio,3169627.74,1827851.77
6,"Music, Movies and Audio Books",3131006.44,1909259.17
4,Games and Toys,724829.43,396668.77


In [9]:
# Check monthly coverage and validation totals against the expected reporting period.
df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.month

monthly_coverage = (
    df.dropna(subset=["Order Date"])
      .groupby(["Year", "Month"])
      .agg(
          Revenue=("Revenue USD", "sum"),
          Orders=("Order Number", "nunique"),
          Sales_Lines=("Order Number", "size")
      )
      .reset_index()
)

display(monthly_coverage.tail(15))

,Year,Month,Revenue,Orders,Sales_Lines
47,2019,12,2477295.85,1250,2967
48,2020,1,2068951.12,1027,2457
49,2020,2,2227379.57,1124,2652
50,2020,3,674990.66,332,775
51,2020,4,217642.10,80,211
52,2020,5,878026.63,455,1107
53,2020,6,763883.23,357,863
54,2020,7,512805.18,253,609
55,2020,8,416647.08,201,488
56,2020,9,380431.52,180,409
